In [ ]:
import pickle
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo


# LOAD DATA

def load_data():
    print("Loading dataset...")

    bank_marketing = fetch_ucirepo(id=222)

    X = bank_marketing.data.features.copy()
    y = bank_marketing.data.targets.copy()

    print(f"Dataset loaded: {X.shape[0]} samples, {X.shape[1]} features.")

    return X, y


def encode_target(y):

    return (
        y.iloc[:, 0]
        .map({"yes": 1, "no": 0})
        .to_numpy(dtype=int)
    )


def train_test_split_df(X,
                        y,
                        test_size=0.2,
                        random_seed=42):

    rng = np.random.default_rng(random_seed)

    idx = rng.permutation(len(X))

    split = int(len(X) * (1 - test_size))

    train_idx = idx[:split]
    test_idx = idx[split:]

    return (
        X.iloc[train_idx].reset_index(drop=True),
        X.iloc[test_idx].reset_index(drop=True),
        y.iloc[train_idx].reset_index(drop=True),
        y.iloc[test_idx].reset_index(drop=True),
    )


# PREPROCESSOR

class MixedNBPreprocessor:

    def __init__(self):

        self.num_cols = None
        self.cat_cols = None

        self.num_means = None
        self.num_stds = None

        self.cat_modes = {}

        self.dummy_cols = None

    def fit(self, X):

        X = X.copy()

        # replace unknown
        X = X.replace("unknown", np.nan)

        # numerical / categorical split
        self.num_cols = (
            X.select_dtypes(include=[np.number])
            .columns
            .tolist()
        )

        self.cat_cols = (
            X.select_dtypes(exclude=[np.number])
            .columns
            .tolist()
        )

        # numerical statistics
        self.num_means = X[self.num_cols].mean()

        self.num_stds = (
            X[self.num_cols]
            .std()
            .replace(0, 1)
        )

        # categorical modes
        for col in self.cat_cols:

            mode = X[col].mode()

            self.cat_modes[col] = (
                mode[0]
                if len(mode) > 0
                else "missing"
            )

        # fit one-hot columns
        X_cat = X[self.cat_cols].copy()

        for col in self.cat_cols:
            X_cat[col] = X_cat[col].fillna(
                self.cat_modes[col]
            )

        X_cat = pd.get_dummies(
            X_cat,
            drop_first=False
        )

        self.dummy_cols = X_cat.columns.tolist()

        return self

    def transform(self, X):

        X = X.copy()

        X = X.replace("unknown", np.nan)

        # NUMERICAL FEATURES

        X_num = X[self.num_cols].copy()

        X_num = X_num.fillna(self.num_means)

        # standardize ONLY numerical features
        X_num = (
            (X_num - self.num_means)
            /
            self.num_stds
        )

        # CATEGORICAL FEATURES

        X_cat = X[self.cat_cols].copy()

        for col in self.cat_cols:
            X_cat[col] = X_cat[col].fillna(
                self.cat_modes[col]
            )

        X_cat = pd.get_dummies(
            X_cat,
            drop_first=False
        )

        # align columns
        for col in self.dummy_cols:

            if col not in X_cat.columns:
                X_cat[col] = 0

        X_cat = X_cat[self.dummy_cols]

        return (
            X_num.to_numpy(dtype=float),
            X_cat.to_numpy(dtype=int)
        )

    def fit_transform(self, X):

        self.fit(X)

        return self.transform(X)


# MIXED NAIVE BAYES

class MixedNaiveBayesScratch:

    def __init__(self,
                 var_smoothing=1e-6,
                 alpha=1.0):

        self.var_smoothing = var_smoothing
        self.alpha = alpha

    def fit(self,
            X_num,
            X_cat,
            y):

        self.classes = np.unique(y)

        self.priors = {}

        self.num_means = {}
        self.num_vars = {}

        self.cat_probs = {}

        n = len(y)

        for c in self.classes:

            Xn_c = X_num[y == c]
            Xc_c = X_cat[y == c]

            # CLASS PRIOR

            self.priors[c] = len(Xn_c) / n

            # GAUSSIAN NB (NUMERICAL)

            self.num_means[c] = (
                Xn_c.mean(axis=0)
            )

            self.num_vars[c] = (
                Xn_c.var(axis=0)
                +
                self.var_smoothing
            )

            # BERNOULLI NB (CATEGORICAL)

            self.cat_probs[c] = (
                (Xc_c.sum(axis=0) + self.alpha)
                /
                (Xc_c.shape[0] + 2 * self.alpha)
            )

        return self

    def _gaussian_log_likelihood(self,
                                 c,
                                 x):

        mean = self.num_means[c]

        var = self.num_vars[c]

        return -0.5 * np.sum(
            np.log(2 * np.pi * var)
            +
            ((x - mean) ** 2) / var
        )

    def _bernoulli_log_likelihood(self,
                                  c,
                                  x):

        p = np.clip(
            self.cat_probs[c],
            1e-12,
            1 - 1e-12
        )

        return np.sum(
            x * np.log(p)
            +
            (1 - x) * np.log(1 - p)
        )

    def predict_proba(self,
                      X_num,
                      X_cat):

        log_probs = []

        for i in range(X_num.shape[0]):

            scores = []

            for c in self.classes:

                score = (
                    np.log(self.priors[c])
                    +
                    self._gaussian_log_likelihood(
                        c,
                        X_num[i]
                    )
                    +
                    self._bernoulli_log_likelihood(
                        c,
                        X_cat[i]
                    )
                )

                scores.append(score)

            log_probs.append(scores)

        log_probs = np.array(log_probs)

        # numerical stability
        log_probs -= np.max(
            log_probs,
            axis=1,
            keepdims=True
        )

        probs = np.exp(log_probs)

        probs /= probs.sum(
            axis=1,
            keepdims=True
        )

        return probs[:, 1]

    def predict(self,
                X_num,
                X_cat,
                threshold=0.5):

        probs = self.predict_proba(
            X_num,
            X_cat
        )

        return (
            probs >= threshold
        ).astype(int)


# EVALUATION

def confusion_matrix_values(y_true,
                            y_pred):

    tp = int(np.sum(
        (y_pred == 1)
        &
        (y_true == 1)
    ))

    fp = int(np.sum(
        (y_pred == 1)
        &
        (y_true == 0)
    ))

    tn = int(np.sum(
        (y_pred == 0)
        &
        (y_true == 0)
    ))

    fn = int(np.sum(
        (y_pred == 0)
        &
        (y_true == 1)
    ))

    return tp, fp, tn, fn


def evaluate(y_true,
             y_pred):

    tp, fp, tn, fn = confusion_matrix_values(
        y_true,
        y_pred
    )

    accuracy = (
        (tp + tn)
        /
        (tp + tn + fp + fn)
    )

    precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn > 0
        else 0
    )

    f1 = (
        2 * precision * recall
        /
        (precision + recall)
        if precision + recall > 0
        else 0
    )

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn,
    }


# THRESHOLD SEARCH

def threshold_search(model,
                     X_test_num,
                     X_test_cat,
                     y_test):

    thresholds = np.arange(
        0.01,
        0.61,
        0.01
    )

    best_result = None

    print("\n" + "=" * 72)
    print("  MIXED NAIVE BAYES THRESHOLD SEARCH")
    print("=" * 72)

    print(
        f"  {'Threshold':<10}"
        f"{'Accuracy':<10}"
        f"{'Precision':<10}"
        f"{'Recall':<10}"
        f"{'F1':<10}"
    )

    print("  " + "-" * 60)

    for threshold in thresholds:

        y_pred = model.predict(
            X_test_num,
            X_test_cat,
            threshold=threshold
        )

        metrics = evaluate(
            y_test,
            y_pred
        )

        if (
            best_result is None
            or
            metrics["F1"] > best_result["F1"]
        ):

            best_result = metrics.copy()

            best_result["Threshold"] = threshold

        # print selected thresholds
        if round(threshold, 2) in [
            0.01,
            0.03,
            0.05,
            0.10,
            0.15,
            0.20,
            0.25,
            0.30,
            0.40,
            0.50
        ]:

            print(
                f"  {threshold:<10.2f}"
                f"{metrics['Accuracy']:<10.4f}"
                f"{metrics['Precision']:<10.4f}"
                f"{metrics['Recall']:<10.4f}"
                f"{metrics['F1']:<10.4f}"
            )

    print("  " + "-" * 60)

    print("\nBEST RESULT:")

    for k, v in best_result.items():

        if isinstance(v, float):
            print(f"  {k:<12}: {v:.4f}")
        else:
            print(f"  {k:<12}: {v}")

    print("=" * 72)

    return best_result


# SAVE ARTIFACTS

def save_artifacts(model,
                   preprocessor):

    with open(
        "mixed_nb_model.pkl",
        "wb"
    ) as f:

        pickle.dump(model, f)

    with open(
        "mixed_nb_preprocessor.pkl",
        "wb"
    ) as f:

        pickle.dump(preprocessor, f)

    print("  Model saved.")
    print("  Preprocessor saved.")


# MAIN

def main():

    print("\n" + "=" * 60)
    print("  BANK MARKETING MIXED NAIVE BAYES PIPELINE")
    print("  Gaussian NB + Bernoulli NB")
    print("=" * 60)

    # LOAD DATA

    X_raw, y_raw = load_data()

    # SPLIT BEFORE PREPROCESSING

    print("\nSplitting data before preprocessing...")

    (
        X_train_raw,
        X_test_raw,
        y_train_raw,
        y_test_raw
    ) = train_test_split_df(
        X_raw,
        y_raw,
        test_size=0.2,
        random_seed=42
    )

    y_train = encode_target(y_train_raw)

    y_test = encode_target(y_test_raw)

    print(f"  Train samples: {len(X_train_raw)}")
    print(f"  Test samples : {len(X_test_raw)}")

    # PREPROCESSING

    print("\nPreprocessing...")

    preprocessor = MixedNBPreprocessor()

    X_train_num, X_train_cat = (
        preprocessor.fit_transform(
            X_train_raw
        )
    )

    X_test_num, X_test_cat = (
        preprocessor.transform(
            X_test_raw
        )
    )

    print(f"  Numerical features   : {X_train_num.shape[1]}")
    print(f"  Categorical features : {X_test_cat.shape[1]}")

    # TRAIN MODEL

    print("\nTraining Mixed Naive Bayes...")

    model = MixedNaiveBayesScratch(
        var_smoothing=1e-6,
        alpha=1.0
    )

    model.fit(
        X_train_num,
        X_train_cat,
        y_train
    )

    # DEFAULT THRESHOLD

    print("\nDefault threshold = 0.5")

    y_pred_05 = model.predict(
        X_test_num,
        X_test_cat,
        threshold=0.5
    )

    metrics_05 = evaluate(
        y_test,
        y_pred_05
    )

    for k in [
        "Accuracy",
        "Precision",
        "Recall",
        "F1"
    ]:

        print(
            f"  {k:<10}: "
            f"{metrics_05[k]:.4f}"
        )

    # THRESHOLD SEARCH

    best_result = threshold_search(
        model,
        X_test_num,
        X_test_cat,
        y_test
    )

    # SAVE MODEL

    print("\nSaving artifacts...")

    save_artifacts(
        model,
        preprocessor
    )

    print("\nPIPELINE COMPLETE")


if __name__ == "__main__":
    main()


  BANK MARKETING MIXED NAIVE BAYES PIPELINE
  Gaussian NB + Bernoulli NB
Loading dataset...
Dataset loaded: 45211 samples, 16 features.

Splitting data before preprocessing...
  Train samples: 36168
  Test samples : 9043

Preprocessing...
  Numerical features   : 7
  Categorical features : 40

Training Mixed Naive Bayes...

Default threshold = 0.5
  Accuracy  : 0.8742
  Precision : 0.4688
  Recall    : 0.5005
  F1        : 0.4841

  MIXED NAIVE BAYES THRESHOLD SEARCH
  Threshold Accuracy  Precision Recall    F1        
  ------------------------------------------------------------
  0.01      0.4856    0.1792    0.9381    0.3009    
  0.03      0.6446    0.2328    0.8763    0.3678    
  0.05      0.7097    0.2669    0.8360    0.4046    
  0.10      0.7903    0.3305    0.7573    0.4601    
  0.15      0.8228    0.3686    0.7029    0.4836    
  0.20      0.8409    0.3952    0.6570    0.4935    
  0.25      0.8503    0.4117    0.6270    0.4970    
  0.30      0.8575    0.4253    0.5923  